In [1]:
import pandas as pd
import numpy as np
import time

X_train = pd.read_csv("../data/interim/X_train.csv")
X_test = pd.read_csv("../data/interim/X_test.csv")
y_train = pd.read_csv("../data/interim/y_train.csv").squeeze()
y_test = pd.read_csv("../data/interim/y_test.csv").squeeze()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train distribution:")
print(y_train.value_counts(normalize=True).round(4) * 100)

X_train: (45757, 166)
X_test: (11440, 166)
y_train distribution:
acceptsassignement_bool
1    51.43
0    48.57
Name: proportion, dtype: float64


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

RANDOM_SEED = 42

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_SEED),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_SEED),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_SEED),
}

print("Models to train:", list(models.keys()))


Models to train: ['Logistic Regression', 'Decision Tree', 'Random Forest', 'Extra Trees', 'Gradient Boosting', 'HistGradientBoosting']


In [3]:
results = []

for name, model in models.items():
    print(f"Training {name}...")
    
    start_train = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_train
    
    start_pred = time.time()
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    pred_time = time.time() - start_pred
    
    # Overfitting check: compare train vs test accuracy
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, y_pred)
    
    results.append({
        "Model": name,
        "Accuracy": round(test_acc, 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1": round(f1_score(y_test, y_pred), 4),
        "ROC_AUC": round(roc_auc_score(y_test, y_proba), 4),
        "PR_AUC": round(average_precision_score(y_test, y_proba), 4),
        "Train_Time_sec": round(train_time, 2),
        "Predict_Time_sec": round(pred_time, 4),
        "Train_Accuracy": round(train_acc, 4),
        "Overfit_Gap": round(train_acc - test_acc, 4),
    })
    
    print(f"  Done. Test accuracy: {test_acc:.4f}, ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")

results_df = pd.DataFrame(results).sort_values("ROC_AUC", ascending=False)
print()
print(results_df.to_string(index=False))

Training Logistic Regression...


C:\Users\home\cms-supplier-acceptance-analysis\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Done. Test accuracy: 0.8223, ROC-AUC: 0.8985
Training Decision Tree...
  Done. Test accuracy: 0.8863, ROC-AUC: 0.8862
Training Random Forest...
  Done. Test accuracy: 0.9077, ROC-AUC: 0.9755
Training Extra Trees...
  Done. Test accuracy: 0.9026, ROC-AUC: 0.9723
Training Gradient Boosting...
  Done. Test accuracy: 0.8709, ROC-AUC: 0.9525
Training HistGradientBoosting...
  Done. Test accuracy: 0.9007, ROC-AUC: 0.9701

               Model  Accuracy  Precision  Recall     F1  ROC_AUC  PR_AUC  Train_Time_sec  Predict_Time_sec  Train_Accuracy  Overfit_Gap
       Random Forest    0.9077     0.9059  0.9157 0.9108   0.9755  0.9757            4.03            0.4102          1.0000       0.0923
         Extra Trees    0.9026     0.9073  0.9030 0.9051   0.9723  0.9720            7.12            0.2938          1.0000       0.0974
HistGradientBoosting    0.9007     0.8928  0.9171 0.9048   0.9701  0.9715            5.71            0.1830          0.9108       0.0101
   Gradient Boosting    0.8709

In [4]:
results_df["Selected"] = results_df["Model"] == "HistGradientBoosting"
results_df["Selection_Reason"] = np.where(
    results_df["Model"] == "HistGradientBoosting",
    "Comparable ROC-AUC to Random Forest/Extra Trees (0.9701 vs 0.9755/0.9723) but far lower overfitting (0.0101 gap vs 0.0923/0.0974) - better generalization expected on unseen data",
    ""
)

results_df.to_csv("../reports/validation/model_comparison.csv", index=False)
print(results_df[["Model", "ROC_AUC", "Overfit_Gap", "Selected"]].to_string(index=False))

               Model  ROC_AUC  Overfit_Gap  Selected
       Random Forest   0.9755       0.0923     False
         Extra Trees   0.9723       0.0974     False
HistGradientBoosting   0.9701       0.0101      True
   Gradient Boosting   0.9525      -0.0030     False
 Logistic Regression   0.8985      -0.0053     False
       Decision Tree   0.8862       0.1137     False
